## BARAM 2026 - DeepSets Distribution Diversity

챔피언형 anchor와 오차 상관이 다른 직접 확률회귀 파트너를 만드는 실험입니다.

- 순서가 임의인 AnEn 30개 멤버를 permutation-invariant DeepSets로 인코딩
- 269개 환경 피처를 별도 context encoder에 통과
- 발전량 CF의 5개 분위수를 직접 예측하며 잔차 LightGBM을 사용하지 않음
- anchor는 고정하고 새 모델 비중을 최대 20%로 제한
- full CV뿐 아니라 leave-one-fold-out meta OOF가 개선될 때만 제출 파일 생성
- 직접 FiCR 위치 이동은 과거 검증에서 불안정했으므로 사용하지 않음

### 1. Colab·Drive 설정

In [ ]:
import os, sys, json, hashlib, random, re, time, pickle, platform, subprocess
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "lightgbm", "xgboost",
         "torch", "tqdm", "pyarrow", "openpyxl", "eccodes"],
        check=True,
    )

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgbm
from tqdm.auto import tqdm

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BARAM2026/data")
    BASE_RUN_ROOT = Path("/content/drive/MyDrive/BARAM2026/hyejin_external_champion")
    RUN_ROOT = Path("/content/drive/MyDrive/BARAM2026/hyejin_deepset_distribution_diversity")
else:
    DATA_DIR = Path.cwd() / "data"
    BASE_RUN_ROOT = Path.cwd() / "runs" / "hyejin_external_champion"
    RUN_ROOT = Path.cwd() / "runs" / "hyejin_deepset_distribution_diversity"

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
EXTERNAL_DIR = DATA_DIR / "external"
BASE_CACHE_DIR = BASE_RUN_ROOT / "cache"
CACHE_DIR = RUN_ROOT / "cache"
for path in [RUN_ROOT, EXTERNAL_DIR, BASE_CACHE_DIR, CACHE_DIR]:
    path.mkdir(exist_ok=True, parents=True)

# Open-Meteo 무료 API의 비상업 이용 조건을 팀에서 확인한 뒤 True로 바꿉니다.
ACK_OPEN_METEO_TERMS = False  #@param {type:"boolean"}
FORCE_JMA_FETCH = False
FORCE_ECMWF_FETCH = False
FINAL_GBM_SEEDS = 3
FINAL_DL_SEEDS = 3

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("device:", DEVICE, "/", GPU_NAME)
if IN_COLAB and "A100" not in GPU_NAME.upper():
    print("[주의] A100 런타임인지 확인하세요.")

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
GROUP_SCALES = {1: 0.98, 2: 0.98, 3: 0.97}

REQUIRED = [
    "train/train_labels.csv", "train/ldaps_train.csv", "train/gfs_train.csv",
    "train/scada_vestas_train.csv", "train/scada_unison_train.csv",
    "test/ldaps_test.csv", "test/gfs_test.csv", "info.xlsx", "sample_submission.csv",
]
missing = [rel for rel in REQUIRED if not (DATA_DIR / rel).exists()]
assert not missing, "누락된 공식 파일:\n" + "\n".join(missing)

def sha256_file(path, block=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(block)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print("data    :", DATA_DIR)
print("external:", EXTERNAL_DIR)
print("base cache:", BASE_CACHE_DIR)
print("output    :", RUN_ROOT)

### 2. 기본 데이터셋 생성 함수

In [ ]:
# 1. v10 데이터셋
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def dms_to_decimal(s):
    pattern = r"(\d+)°(\d+)'([\d.]+)\"([NS])\s+(\d+)°(\d+)'([\d.]+)\"([EW])"
    m = re.match(pattern, s.strip())
    d1, m1, s1, hemi1, d2, m2, s2, hemi2 = m.groups()
    lat = float(d1) + float(m1) / 60 + float(s1) / 3600
    if hemi1 == "S": lat = -lat
    lon = float(d2) + float(m2) / 60 + float(s2) / 3600
    if hemi2 == "W": lon = -lon
    return lat, lon

def find_nearest_grid(weather_df, coord_df, name_col, lat_col="lat", lon_col="lon"):
    grids = weather_df[["grid_id", "latitude", "longitude"]].drop_duplicates()
    mapping = {}
    for _, row in coord_df.iterrows():
        dists = haversine(row[lat_col], row[lon_col], grids["latitude"], grids["longitude"])
        mapping[row[name_col]] = grids.loc[dists.idxmin(), "grid_id"]
    return mapping

def add_wind_features(df, uv_pairs):
    df = df.copy()
    for u_col, v_col, height in uv_pairs:
        df[f"ws_{height}"] = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)
        wd_rad = np.arctan2(df[v_col], df[u_col])
        df[f"wd_sin_{height}"] = np.sin(wd_rad)
        df[f"wd_cos_{height}"] = np.cos(wd_rad)
    return df

def build_group_weather(weather_df, group_grid_map, prefix):
    weather_df = weather_df.copy()
    weather_df["forecast_kst_dtm"] = pd.to_datetime(weather_df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in weather_df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    frames = []
    for group, grid_id in group_grid_map.items():
        sub = weather_df.loc[weather_df["grid_id"] == grid_id, ["forecast_kst_dtm", *value_cols]].copy()
        sub = sub.rename(columns={c: f"{prefix}_{c}" for c in value_cols})
        sub["kpx_group"] = int(group)
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

def add_forecast_disagreement(df):
    df = df.copy()
    df["ws_disagreement"] = np.abs(df["ldaps_ws_10m"] - df["gfs_ws_10m"])
    df["wd_agreement"] = df["ldaps_wd_cos_10m"] * df["gfs_wd_cos_10m"] + df["ldaps_wd_sin_10m"] * df["gfs_wd_sin_10m"]
    return df

def add_seasonal_interaction(df):
    df = df.copy()
    month = pd.to_datetime(df["forecast_kst_dtm"]).dt.month
    df["high_variance_season"] = month.isin([1, 2, 3, 4, 5, 7]).astype(int)
    df["ws_x_season"] = df["ldaps_ws_10m"] * df["high_variance_season"]
    df["disagreement_x_season"] = df["ws_disagreement"] * df["high_variance_season"]
    return df

def build_datasets():
    train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
    sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
    ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
    gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
    ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
    gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")
    train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
    sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

    raw = pd.read_excel(DATA_DIR / "info.xlsx", sheet_name="info", header=None)
    info = raw.iloc[4:21].copy()
    info.columns = raw.iloc[3]
    info = info.reset_index(drop=True)
    info = info.rename(columns={"좌표(Google)": "coord", "KPX그룹": "kpx_group", "설비용량(MW)": "capacity_mw"})
    info["kpx_group"] = info["kpx_group"].ffill().astype(int)
    coords = info["coord"].apply(dms_to_decimal)
    info["lat"] = coords.apply(lambda x: x[0])
    info["lon"] = coords.apply(lambda x: x[1])
    assert info.groupby("kpx_group")["capacity_mw"].sum().tolist() == [21.6, 21.6, 21.0]

    group_coords = info.groupby("kpx_group")[["lat", "lon"]].mean().reset_index()
    group_coords = group_coords.rename(columns={"kpx_group": "KPX그룹"})
    ldaps_group_grid = find_nearest_grid(ldaps_train, group_coords, "KPX그룹")
    gfs_group_grid = find_nearest_grid(gfs_train, group_coords, "KPX그룹")

    LDAPS_UV_PAIRS = [("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m")]
    GFS_UV_PAIRS = [
        ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m"),
        ("heightAboveGround_80_u", "heightAboveGround_80_v", "80m"),
        ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "100m"),
        ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "pbl"),
    ]
    ldaps_train = add_wind_features(ldaps_train, LDAPS_UV_PAIRS)
    ldaps_test = add_wind_features(ldaps_test, LDAPS_UV_PAIRS)
    gfs_train = add_wind_features(gfs_train, GFS_UV_PAIRS)
    gfs_test = add_wind_features(gfs_test, GFS_UV_PAIRS)
    for df in [gfs_train, gfs_test]:
        df["wind_shear_100_10"] = df["ws_100m"] - df["ws_10m"]
        df["wind_shear_80_10"] = df["ws_80m"] - df["ws_10m"]

    ldaps_group_train = build_group_weather(ldaps_train, ldaps_group_grid, "ldaps")
    ldaps_group_test = build_group_weather(ldaps_test, ldaps_group_grid, "ldaps")
    gfs_group_train = build_group_weather(gfs_train, gfs_group_grid, "gfs")
    gfs_group_test = build_group_weather(gfs_test, gfs_group_grid, "gfs")

    train_labels_long = train_labels.melt(id_vars=["kst_dtm"], value_vars=TARGET_COLS,
                                          var_name="kpx_group_col", value_name="target")
    train_labels_long["kpx_group"] = train_labels_long["kpx_group_col"].str.extract(r"(\d+)").astype(int)
    train_labels_long = train_labels_long.rename(columns={"kst_dtm": "forecast_kst_dtm"}).drop(columns=["kpx_group_col"])

    test_base = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
    test_long = pd.concat([test_base.assign(kpx_group=g) for g in [1, 2, 3]], ignore_index=True)

    train_df = train_labels_long.merge(ldaps_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left") \
                                  .merge(gfs_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left")
    test_df = test_long.merge(ldaps_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left") \
                        .merge(gfs_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left")
    assert train_df.shape[0] == train_labels_long.shape[0]

    train_df = add_seasonal_interaction(add_forecast_disagreement(train_df))
    test_df = add_seasonal_interaction(add_forecast_disagreement(test_df))
    weather_cols = [c for c in train_df.columns if c not in ["forecast_kst_dtm", "kpx_group", "target"]]
    print(f"v10 피처 개수: {len(weather_cols)}")
    return train_df, test_df, weather_cols, sample_submission, info

### 3. 평가 지표·검증 fold

In [ ]:
# 2. 공통 유틸 — metric, 편향보정, fold
def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"], out["day"], out["hour"] = dt.dt.month, dt.dt.day, dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2*np.pi*out["hour"]/24); out["hour_cos"] = np.cos(2*np.pi*out["hour"]/24)
    out["month_sin"] = np.sin(2*np.pi*out["month"]/12); out["month_cos"] = np.cos(2*np.pi*out["month"]/12)
    return out

def metric(answer_df, pred_df):
    group_nmae, group_ficr = [], []
    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float); forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]; valid = actual >= capacity * 0.10
        actual, forecast = actual[valid], forecast[valid]
        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))
        unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
        group_ficr.append(np.sum(actual * unit_price) / np.sum(actual * 4.0))
    one_minus_nmae = 1 - np.mean(group_nmae); ficr = np.mean(group_ficr)
    return 0.5 * one_minus_nmae + 0.5 * ficr, one_minus_nmae, ficr

def make_sample_weight(y, capacity, low_weight=0.05):
    return np.where(y >= capacity * 0.10, 1.0, low_weight)

def apply_rolling_smoothing(pred_series, window=3, alpha=0.5):
    rolling = pred_series.rolling(window, center=True, min_periods=1).mean()
    return alpha * pred_series + (1 - alpha) * rolling

def apply_bias_correction_v2(pred, ws, group, bias_map):
    pred = pred.copy()
    if group not in bias_map: return pred
    for interval, bias in bias_map[group].items():
        mask = (ws > interval.left) & (ws <= interval.right)
        pred[mask] = pred[mask] * (1 - bias)
    return pred

def lgbm_params(n_estimators, n_jobs, seed, objective="mae"):
    # GPU(OpenCL) 벤치마크: CPU 143s -> GPU 74s (60k행x352열, 500트리 기준)
    return dict(n_estimators=n_estimators, max_depth=6, learning_rate=0.03, num_leaves=31,
                min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                random_state=seed, n_jobs=n_jobs, verbose=-1, objective=objective, device="gpu")

def prepare_matrices(tr_g, va_g, weather_cols):
    tr_feat = pd.concat([calendar_features(tr_g["forecast_kst_dtm"]).reset_index(drop=True),
                         tr_g[weather_cols].reset_index(drop=True)], axis=1)
    va_feat = pd.concat([calendar_features(va_g["forecast_kst_dtm"]).reset_index(drop=True),
                         va_g[weather_cols].reset_index(drop=True)], axis=1)
    imputer = SimpleImputer(strategy="median")
    X_tr = pd.DataFrame(imputer.fit_transform(tr_feat), columns=tr_feat.columns)
    X_va = pd.DataFrame(imputer.transform(va_feat), columns=va_feat.columns)
    return X_tr, X_va

FOLDS_Q = [
    {"name": "fold_2024Q1", "train_end": pd.Timestamp("2024-01-01"), "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2024-04-01")},
    {"name": "fold_2024Q2", "train_end": pd.Timestamp("2024-04-01"), "valid_start": pd.Timestamp("2024-04-01 01:00:00"), "valid_end": pd.Timestamp("2024-07-01")},
    {"name": "fold_2024Q3", "train_end": pd.Timestamp("2024-07-01"), "valid_start": pd.Timestamp("2024-07-01 01:00:00"), "valid_end": pd.Timestamp("2024-10-01")},
    {"name": "fold_2024Q4", "train_end": pd.Timestamp("2024-10-01"), "valid_start": pd.Timestamp("2024-10-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01")},
]
HOLD_FOLD = {"name": "hold_2024full", "train_end": pd.Timestamp("2024-01-01"),
             "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01")}
FOLD_LIST = [(f["name"], f) for f in FOLDS_Q] + [(HOLD_FOLD["name"], HOLD_FOLD)]

COARSE_BINS = [0, 11, 13, 100]

def estimate_bias_data(train_df, weather_cols, tr_end, n_estimators, n_jobs,
                       low_weight=0.05, holdout_ratio=0.25, min_fit_days=90):
    out = {}
    for group in [1, 2, 3]:
        capacity = CAPACITY_KWH[f"kpx_group_{group}"]
        g_data = train_df[(train_df["kpx_group"] == group) & (train_df["forecast_kst_dtm"] <= tr_end)]
        g_data = g_data[g_data["target"].notna()]
        if len(g_data) == 0: continue
        first_date = g_data["forecast_kst_dtm"].min()
        available_days = max((tr_end - first_date).days, 1)
        holdout_days = min(int(available_days * holdout_ratio), 180)
        if available_days - holdout_days < min_fit_days:
            holdout_days = max(0, available_days - min_fit_days)
        tr_sub_end = tr_end - pd.Timedelta(days=holdout_days)
        fit_g = g_data[g_data["forecast_kst_dtm"] <= tr_sub_end]
        check_g = g_data[g_data["forecast_kst_dtm"] > tr_sub_end]
        if len(fit_g) == 0 or len(check_g) == 0: continue
        fit_feat = pd.concat([calendar_features(fit_g["forecast_kst_dtm"]).reset_index(drop=True),
                              fit_g[weather_cols].reset_index(drop=True)], axis=1)
        check_feat = pd.concat([calendar_features(check_g["forecast_kst_dtm"]).reset_index(drop=True),
                                check_g[weather_cols].reset_index(drop=True)], axis=1)
        imputer = SimpleImputer(strategy="median")
        X_fit = pd.DataFrame(imputer.fit_transform(fit_feat), columns=fit_feat.columns)
        X_check = pd.DataFrame(imputer.transform(check_feat), columns=check_feat.columns)
        w_fit = make_sample_weight(fit_g["target"].to_numpy(), capacity, low_weight)
        model = lgb.LGBMRegressor(**lgbm_params(n_estimators, n_jobs, SEED))
        model.fit(X_fit, fit_g["target"], sample_weight=w_fit)
        pred_check = model.predict(X_check)
        check_g = check_g.copy()
        check_g["signed_error_rate"] = (pred_check - check_g["target"]) / capacity
        check_g = check_g.loc[check_g["target"] >= capacity * 0.10]
        if len(check_g) == 0: continue
        out[group] = check_g[["ldaps_ws_10m", "signed_error_rate"]]
    return out

def bias_map_from_data(bias_data, bins_by_group):
    bias_map = {}
    for group, df in bias_data.items():
        ws_bin = pd.cut(df["ldaps_ws_10m"], bins=bins_by_group[group])
        bias_map[group] = df.groupby(ws_bin, observed=True)["signed_error_rate"].mean().to_dict()
    return bias_map

### 4. 물리·시계열·공간 피처

In [ ]:
# 3. v10TPS 피처 (T+P+S 블록)
def add_physics(df):
    df = df.copy()
    rho = df["ldaps_surface_0_sp"] / (287.05 * df["ldaps_heightAboveGround_2_t"])
    df["phy_density"] = rho
    df["phy_ws10_cu"] = df["ldaps_ws_10m"] ** 3
    df["phy_ws100_cu"] = df["gfs_ws_100m"] ** 3
    df["phy_rho_ws100_cu"] = rho * df["phy_ws100_cu"]
    df["phy_pc100"] = np.clip(df["gfs_ws_100m"], 0, 12) ** 3 / 12 ** 3
    ws50max = np.hypot(df["ldaps_heightAboveGround_50_50MUmax"], df["ldaps_heightAboveGround_50_50MVmax"])
    ws50min = np.hypot(df["ldaps_heightAboveGround_50_50MUmin"], df["ldaps_heightAboveGround_50_50MVmin"])
    df["phy_ws50max"] = ws50max
    df["phy_gust_range"] = ws50max - ws50min
    df["phy_gust_over_ws"] = ws50max / (df["ldaps_ws_10m"] + 0.5)
    df["phy_gfs_gust_factor"] = df["gfs_surface_0_gust"] / (df["gfs_ws_10m"] + 0.5)
    ws850 = np.hypot(df["gfs_isobaricInhPa_850_u"], df["gfs_isobaricInhPa_850_v"])
    df["phy_ws850"] = ws850
    df["phy_shear_850_100"] = ws850 - df["gfs_ws_100m"]
    cols = [c for c in df.columns if c.startswith("phy_")]
    return df, cols

def add_temporal(df):
    df = df.sort_values(["kpx_group", "forecast_kst_dtm"]).copy()
    for src in ["ldaps_ws_10m", "gfs_ws_100m"]:
        g = df.groupby("kpx_group")[src]
        p = "tmp_" + ("l10" if src.startswith("ldaps") else "g100")
        df[f"{p}_r3_mean"] = g.transform(lambda s: s.rolling(3, center=True, min_periods=1).mean())
        df[f"{p}_r3_std"] = g.transform(lambda s: s.rolling(3, center=True, min_periods=1).std())
        df[f"{p}_r7_mean"] = g.transform(lambda s: s.rolling(7, center=True, min_periods=1).mean())
        df[f"{p}_r7_std"] = g.transform(lambda s: s.rolling(7, center=True, min_periods=1).std())
        df[f"{p}_d1"] = g.transform(lambda s: s.diff(1))
        df[f"{p}_dm1"] = g.transform(lambda s: s.diff(-1))
        df[f"{p}_d3"] = g.transform(lambda s: s.diff(3))
        df[f"{p}_dm3"] = g.transform(lambda s: s.diff(-3))
    day_key = (pd.to_datetime(df["forecast_kst_dtm"]) - pd.Timedelta(hours=1)).dt.date
    day_grp = df.groupby([df["kpx_group"], day_key])["ldaps_ws_10m"]
    df["tmp_day_mean"] = day_grp.transform("mean")
    df["tmp_day_max"] = day_grp.transform("max")
    df["tmp_day_std"] = day_grp.transform("std")
    cols = [c for c in df.columns if c.startswith("tmp_")]
    return df, cols

def build_spatial_frames(info):
    centroids = info.groupby("kpx_group")[["lat", "lon"]].mean()
    out = {}
    for split, ld_path, gf_path in [("train", TRAIN_DIR / "ldaps_train.csv", TRAIN_DIR / "gfs_train.csv"),
                                    ("test", TEST_DIR / "ldaps_test.csv", TEST_DIR / "gfs_test.csv")]:
        ld = pd.read_csv(ld_path, encoding="utf-8-sig",
                         usecols=["forecast_kst_dtm", "grid_id", "latitude", "longitude",
                                  "heightAboveGround_10_10u", "heightAboveGround_10_10v"])
        ld["forecast_kst_dtm"] = pd.to_datetime(ld["forecast_kst_dtm"])
        ld["ws"] = np.hypot(ld["heightAboveGround_10_10u"], ld["heightAboveGround_10_10v"])
        grids = ld[["grid_id", "latitude", "longitude"]].drop_duplicates().set_index("grid_id")
        frames = []
        for group in [1, 2, 3]:
            d = haversine(centroids.loc[group, "lat"], centroids.loc[group, "lon"], grids["latitude"], grids["longitude"])
            near = d.nsmallest(4).index
            sub = ld[ld["grid_id"].isin(near)].groupby("forecast_kst_dtm")["ws"]
            agg = pd.DataFrame({"spa_l10_mean": sub.mean(), "spa_l10_std": sub.std(),
                                "spa_l10_range": sub.max() - sub.min()}).reset_index()
            agg["kpx_group"] = group
            frames.append(agg)
        ldaps_spa = pd.concat(frames, ignore_index=True)
        gf = pd.read_csv(gf_path, encoding="utf-8-sig",
                         usecols=["forecast_kst_dtm", "grid_id", "heightAboveGround_100_100u", "heightAboveGround_100_100v"])
        gf["forecast_kst_dtm"] = pd.to_datetime(gf["forecast_kst_dtm"])
        gf["ws100"] = np.hypot(gf["heightAboveGround_100_100u"], gf["heightAboveGround_100_100v"])
        sub = gf.groupby("forecast_kst_dtm")["ws100"]
        gfs_spa = pd.DataFrame({"spa_g100_mean": sub.mean(), "spa_g100_std": sub.std()}).reset_index()
        out[split] = (ldaps_spa, gfs_spa)
    return out

def build_v11(info):
    train_df, test_df, weather_cols_v10, sample_submission, _ = build_datasets()
    spa = build_spatial_frames(info)
    dfs = {}
    for split, df in [("train", train_df), ("test", test_df)]:
        df, phy_cols = add_physics(df)
        df, tmp_cols = add_temporal(df)
        ldaps_spa, gfs_spa = spa[split]
        df = df.merge(ldaps_spa, on=["forecast_kst_dtm", "kpx_group"], how="left")
        df = df.merge(gfs_spa, on="forecast_kst_dtm", how="left")
        dfs[split] = df
    spa_cols = ["spa_l10_mean", "spa_l10_std", "spa_l10_range", "spa_g100_mean", "spa_g100_std"]
    v10tps_cols = weather_cols_v10 + tmp_cols + phy_cols + spa_cols
    return dfs["train"], dfs["test"], v10tps_cols, sample_submission

### 5. 다격자 편차 피처

In [ ]:
# 4. spd 다격자 편차 피처
SPD_PATH = BASE_CACHE_DIR / "spatial_spd.parquet"
LD_VARS = {
    "ws10": lambda d: np.hypot(d["heightAboveGround_10_10u"], d["heightAboveGround_10_10v"]),
    "u10": lambda d: d["heightAboveGround_10_10u"], "v10": lambda d: d["heightAboveGround_10_10v"],
    "ws50x": lambda d: np.hypot(d["heightAboveGround_50_50MUmax"], d["heightAboveGround_50_50MVmax"]),
    "wsbl": lambda d: np.hypot(d["heightAboveGround_5_XBLWS"], d["heightAboveGround_5_YBLWS"]),
    "t2": lambda d: d["heightAboveGround_2_t"], "sp": lambda d: d["surface_0_sp"], "blh": lambda d: d["etc_0_blh"],
}
LD_SRC = ["heightAboveGround_10_10u", "heightAboveGround_10_10v", "heightAboveGround_50_50MUmax",
          "heightAboveGround_50_50MVmax", "heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS",
          "heightAboveGround_2_t", "surface_0_sp", "etc_0_blh"]
GF_VARS = {
    "ws10": lambda d: np.hypot(d["heightAboveGround_10_10u"], d["heightAboveGround_10_10v"]),
    "ws100": lambda d: np.hypot(d["heightAboveGround_100_100u"], d["heightAboveGround_100_100v"]),
    "u100": lambda d: d["heightAboveGround_100_100u"], "v100": lambda d: d["heightAboveGround_100_100v"],
    "gust": lambda d: d["surface_0_gust"],
    "ws850": lambda d: np.hypot(d["isobaricInhPa_850_u"], d["isobaricInhPa_850_v"]),
    "t850": lambda d: d["isobaricInhPa_850_t"], "prmsl": lambda d: d["meanSea_0_prmsl"],
}
GF_SRC = ["heightAboveGround_10_10u", "heightAboveGround_10_10v", "heightAboveGround_100_100u",
          "heightAboveGround_100_100v", "surface_0_gust", "isobaricInhPa_850_u",
          "isobaricInhPa_850_v", "isobaricInhPa_850_t", "meanSea_0_prmsl"]

def _load(paths, src_cols):
    frames = []
    for p in paths:
        d = pd.read_csv(p, encoding="utf-8-sig", usecols=["forecast_kst_dtm", "grid_id", "latitude", "longitude", *src_cols])
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df

def _pivot(df, fn, master_idx, gid):
    v = df[["forecast_kst_dtm", "grid_id"]].copy()
    v["val"] = np.asarray(fn(df), dtype=np.float64)
    M = v.pivot_table(index="forecast_kst_dtm", columns="grid_id", values="val", aggfunc="mean")
    M = M.reindex(index=master_idx, columns=gid)
    row_mean = M.mean(axis=1)
    return M.apply(lambda c: c.fillna(row_mean))

def build_spatial(prefix, paths, src_cols, var_defs):
    df = _load(paths, src_cols)
    grids = df[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id")
    gid = grids["grid_id"].to_numpy()
    master_idx = pd.Index(np.sort(df["forecast_kst_dtm"].unique()), name="forecast_kst_dtm")
    spd_cols = {}
    for name, fn in var_defs.items():
        M = _pivot(df, fn, master_idx, gid)
        V = M.to_numpy()
        mean = V.mean(axis=1)
        spd_cols[f"sd{prefix}_{name}_mean"] = mean
        for j, g in enumerate(gid):
            spd_cols[f"sd{prefix}_{name}_g{int(g)}"] = V[:, j] - mean
    return pd.DataFrame(spd_cols, index=master_idx).reset_index()

def build_spd(force=False):
    if not force and SPD_PATH.exists():
        spd = pd.read_parquet(SPD_PATH)
    else:
        t0 = time.time()
        ld_spd = build_spatial("L", [TRAIN_DIR / "ldaps_train.csv", TEST_DIR / "ldaps_test.csv"], LD_SRC, LD_VARS)
        gf_spd = build_spatial("G", [TRAIN_DIR / "gfs_train.csv", TEST_DIR / "gfs_test.csv"], GF_SRC, GF_VARS)
        spd = ld_spd.merge(gf_spd, on="forecast_kst_dtm", how="outer")
        spd.to_parquet(SPD_PATH, index=False)
        print(f"spd 캐시 생성 완료 ({time.time()-t0:.0f}s)")
    spd_cols = [c for c in spd.columns if c != "forecast_kst_dtm"]
    return spd, spd_cols

### 6. JMA 외부 예보

In [ ]:
# 5. JMA 외부예보 수집
import requests

JMA_PATH = EXTERNAL_DIR
JMA_PATH.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"
MODELS = {"jma_gsm": ["2022", "2023", "2024", "2025"], "jma_msm": ["2023", "2024", "2025"]}
HOURLY_VARS = ["wind_speed_10m", "wind_direction_10m", "temperature_2m", "surface_pressure"]
LEADS = ["previous_day2"]

def fetch_json(url, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=120)
            d = r.json()
            if "hourly" in d:
                return d
            if "reason" in d:
                raise RuntimeError(f"API error: {d['reason']}")
        except Exception as e:
            print(f"  재시도 {attempt+1}: {e}")
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"fetch 실패: {url[:120]}")

def get_centroids(info):
    return info.groupby("kpx_group")[["lat", "lon"]].mean()

def fetch_jma_prevruns(info, force=False):
    force = force or FORCE_JMA_FETCH
    out_path = JMA_PATH / "jma_prevruns.csv"
    if not ACK_OPEN_METEO_TERMS:
        raise RuntimeError(
            "Open-Meteo 이용 조건을 팀에서 확인한 뒤 1번 셀의 "
            "ACK_OPEN_METEO_TERMS=True로 바꾸세요."
        )
    if not force and out_path.exists():
        print(f"JMA 캐시 로드: {out_path}")
        return pd.read_csv(out_path, encoding="utf-8-sig", parse_dates=["forecast_kst_dtm"])
    centroids = get_centroids(info)
    hourly_param = ",".join(f"{v}_{lead}" for v in HOURLY_VARS for lead in LEADS)
    frames = []
    for group in [1, 2, 3]:
        lat = round(float(centroids.loc[group, "lat"]), 4)
        lon = round(float(centroids.loc[group, "lon"]), 4)
        for model, years in MODELS.items():
            for year in years:
                url = (f"{BASE_URL}?latitude={lat}&longitude={lon}&models={model}"
                       f"&hourly={hourly_param}&wind_speed_unit=ms"
                       f"&start_date={year}-01-01&end_date={year}-12-31&timezone=Asia%2FSeoul")
                d = fetch_json(url)
                h = d["hourly"]
                df = pd.DataFrame({"forecast_kst_dtm": pd.to_datetime(h["time"])})
                for v in HOURLY_VARS:
                    for lead in LEADS:
                        short = {"previous_day2": "d2", "previous_day1": "d1"}[lead]
                        df[f"{v}_{short}"] = h.get(f"{v}_{lead}")
                df["kpx_group"] = group
                df["model"] = model
                frames.append(df)
                time.sleep(1)
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(out_path, index=False, encoding="utf-8-sig")
    jma_meta = {
        "source": "Open-Meteo Previous Runs API",
        "source_url": "https://open-meteo.com/en/docs/previous-runs-api",
        "terms_url": "https://open-meteo.com/en/terms",
        "lead": "previous_day2",
        "models": MODELS,
        "variables": HOURLY_VARS,
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (JMA_PATH / "jma_fetch_meta.json").write_text(
        json.dumps(jma_meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Saved: {out_path} ({len(out)}행)")
    return out

def build_jma_frame(jma_df):
    frames = []
    for model, tag in [("jma_gsm", "jgsm"), ("jma_msm", "jmsm")]:
        sub = jma_df[jma_df["model"] == model].copy()
        wd_rad = np.deg2rad(sub["wind_direction_10m_d2"])
        out = pd.DataFrame({
            "forecast_kst_dtm": sub["forecast_kst_dtm"], "kpx_group": sub["kpx_group"],
            f"{tag}_ws": sub["wind_speed_10m_d2"], f"{tag}_wdsin": np.sin(wd_rad), f"{tag}_wdcos": np.cos(wd_rad),
        })
        if model == "jma_gsm":
            out[f"{tag}_t2m"] = sub["temperature_2m_d2"]
            out[f"{tag}_sp"] = sub["surface_pressure_d2"]
        frames.append(out)
    return frames[0].merge(frames[1], on=["forecast_kst_dtm", "kpx_group"], how="outer")

JG_COLS = ["jgsm_ws", "jgsm_wdsin", "jgsm_wdcos", "jgsm_t2m", "jgsm_sp", "jgsm_dis"]
JM_COLS = ["jmsm_ws", "jmsm_wdsin", "jmsm_wdcos", "jmsm_dis"]

### 7. ECMWF HRES 외부 예보

In [ ]:
# 5. ECMWF IFS HRES 외부예보 수집 (open-data S3 아카이브, D-2 12z oper 런)
#   wonwoo 브랜치 exp054_fetch_ecmwf.py / exp054_gate.py 이식.
#   - 규정: D-2일 12z 런 사용 → 예측기준시점(전날 14시 KST) 이전 발행 완료.
#   - Windows에서는 eccodes DLL이 앱 제어 정책에 막혀 fetch/decode를 분리했었지만,
#     이 환경(Linux)에서는 eccodes가 pip만으로 바로 동작해 한 번에 처리한다.
#   - GBM 전용 피처 4개: ec_ws, ec_wdsin, ec_wdcos, ec_dis (DL은 안 받음 — exp-040 교훈:
#     같은 정보를 GBM·DL 양쪽에 주면 오히려 탈상관이 죽어 상쇄된다).
import json
import eccodes as _eccodes
from concurrent.futures import ThreadPoolExecutor, as_completed

EC_PATH = EXTERNAL_DIR
EC_PATH.mkdir(parents=True, exist_ok=True)
EC_RAW_PARQUET = EC_PATH / "ecmwf_prevruns.parquet"
EC_S3 = "https://ecmwf-forecasts.s3.eu-central-1.amazonaws.com"
EC_STEPS = list(range(27, 52, 3))          # 27~51h, 3시간 간격 9개
EC_START, EC_END = "2023-01-18", "2025-12-30"   # 커버리지 시작(아카이브 실측) ~ 테스트기간 D-2 끝

_ec_sess = requests.Session()
_ec_sess.headers["User-Agent"] = "baram2026-simplified"

def _ec_http_get(url, rng=None, retries=6):
    # S3 아카이브가 간헐적으로 503(SlowDown/일시 과부하)을 반환함 — 소규모 스모크 테스트에서
    # 동일 URL이 재시도 시 바로 200으로 성공하는 걸 확인, 재시도 강화로 대응 (지수 백오프, 최대 20초).
    for a in range(retries):
        try:
            h = {"Range": f"bytes={rng[0]}-{rng[0]+rng[1]-1}"} if rng else {}
            r = _ec_sess.get(url, headers=h, timeout=60)
            if r.status_code in (200, 206):
                return r.content
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep(min(2 * (2 ** a), 20))
    return None

def _ec_bilinear(msg_bytes, pts):
    """GRIB 메시지에서 각 (lat, lon) 지점의 이중선형 보간값"""
    gid = _eccodes.codes_new_from_message(msg_bytes)
    try:
        ni = _eccodes.codes_get(gid, "Ni"); nj = _eccodes.codes_get(gid, "Nj")
        lat0 = _eccodes.codes_get(gid, "latitudeOfFirstGridPointInDegrees")
        lon0 = _eccodes.codes_get(gid, "longitudeOfFirstGridPointInDegrees")
        dlat = _eccodes.codes_get(gid, "jDirectionIncrementInDegrees")
        dlon = _eccodes.codes_get(gid, "iDirectionIncrementInDegrees")
        jpos = _eccodes.codes_get(gid, "jScansPositively")
        vals = _eccodes.codes_get_values(gid).reshape(nj, ni)
    finally:
        _eccodes.codes_release(gid)
    slat = dlat if jpos else -dlat
    out = []
    for lat, lon in pts:
        fi = ((lon - lon0) % 360.0) / dlon
        fj = (lat - lat0) / slat
        i0, j0 = int(np.floor(fi)), int(np.floor(fj))
        wi, wj = fi - i0, fj - j0
        j0 = min(max(j0, 0), nj - 2)
        v = ((1 - wi) * (1 - wj) * vals[j0, i0 % ni] + wi * (1 - wj) * vals[j0, (i0 + 1) % ni]
             + (1 - wi) * wj * vals[j0 + 1, i0 % ni] + wi * wj * vals[j0 + 1, (i0 + 1) % ni])
        out.append(float(v))
    return out

def _ec_fetch_day(init_d, pts):
    d = init_d.strftime("%Y%m%d")
    roots = (["12z/0p4-beta/oper", "12z/ifs/0p4-beta/oper", "12z/ifs/0p25/oper"]
             if init_d < pd.Timestamp("2024-02-01") else
             ["12z/ifs/0p25/oper", "12z/ifs/0p4-beta/oper", "12z/0p4-beta/oper"])
    root, idx0 = None, None
    for cand in roots:
        idx0 = _ec_http_get(f"{EC_S3}/{d}/{cand}/{d}120000-27h-oper-fc.index", retries=5)
        if idx0:
            root = cand; break
    if root is None:
        return []
    rows = []
    for s in EC_STEPS:
        raw = idx0 if s == 27 else _ec_http_get(f"{EC_S3}/{d}/{root}/{d}120000-{s}h-oper-fc.index")
        if raw is None:
            continue
        ranges = {}
        for line in raw.decode("utf-8", "ignore").splitlines():
            try:
                e = json.loads(line)
            except json.JSONDecodeError:
                continue
            if e.get("levtype") == "sfc" and e.get("param") in ("10u", "10v"):
                ranges[e["param"]] = (int(e["_offset"]), int(e["_length"]))
        if "10u" not in ranges or "10v" not in ranges:
            continue
        gurl = f"{EC_S3}/{d}/{root}/{d}120000-{s}h-oper-fc.grib2"
        mu = _ec_http_get(gurl, rng=ranges["10u"])
        mv = _ec_http_get(gurl, rng=ranges["10v"])
        if mu is None or mv is None:
            continue
        us = _ec_bilinear(mu, pts)
        vs = _ec_bilinear(mv, pts)
        for g, u, v in zip([1, 2, 3], us, vs):
            rows.append(dict(init_date=d, step=s, group=g, u10=u, v10=v))
    return rows

def fetch_ecmwf_prevruns(info, start=EC_START, end=EC_END, workers=20, force=False):
    force = force or FORCE_ECMWF_FETCH
    # 체크포인트: 실행 환경이 중간에 죽어도(관측됨) 이어서 진행하도록 100일마다 부분 저장.
    inits = pd.date_range(start, end, freq="D")
    all_rows, done_inits = [], set()
    if not force and EC_RAW_PARQUET.exists():
        prev = pd.read_parquet(EC_RAW_PARQUET)
        done_inits = set(prev["init_date"].unique())
        all_rows = prev.to_dict("records")
        if done_inits >= set(d.strftime("%Y%m%d") for d in inits):
            print(f"ECMWF raw 캐시 로드: {len(prev)}행, {len(done_inits)}일 (완료)")
            return prev
        print(f"ECMWF 부분 캐시 로드: {len(prev)}행, {len(done_inits)}일 — 이어서 진행", flush=True)
    todo = [d for d in inits if d.strftime("%Y%m%d") not in done_inits]
    centroids = get_centroids(info)
    pts = [(float(centroids.loc[g, "lat"]), float(centroids.loc[g, "lon"])) for g in [1, 2, 3]]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(_ec_fetch_day, d, pts): d for d in todo}
        n_done, n_ok = 0, 0
        for fut in as_completed(futs):
            r = fut.result()
            n_done += 1
            if r:
                n_ok += 1
                all_rows.extend(r)
            if n_done % 100 == 0:
                el = time.time() - t0
                print(f"  진행 {n_done}/{len(todo)}일 ({el/60:.1f}분, 성공 {n_ok}일, {el/n_done*len(todo)/60:.0f}분 예상)", flush=True)
                pd.DataFrame(all_rows).to_parquet(EC_RAW_PARQUET, index=False)
    out = pd.DataFrame(all_rows)
    out.to_parquet(EC_RAW_PARQUET, index=False)
    ec_meta = {
        "source": "ECMWF Open Data AWS archive",
        "source_url": "https://www.ecmwf.int/en/forecasts/datasets/open-data",
        "run": "D-2 12z IFS HRES",
        "variables": ["10u", "10v"],
        "date_range": [str(start), str(end)],
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (EC_PATH / "ecmwf_fetch_meta.json").write_text(
        json.dumps(ec_meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"ECMWF raw 수집 완료: {len(out)}행, {out['init_date'].nunique()}/{len(inits)}일 ({(time.time()-t0)/60:.1f}분)", flush=True)
    return out

def build_ec_frame(ec_raw):
    """(init_date, step, group, u10, v10) → 시간별 KST ec_ws/ec_wdsin/ec_wdcos (exp054_gate.py build_ec_frame 이식)"""
    ec_raw = ec_raw.copy()
    ec_raw["init_date"] = pd.to_datetime(ec_raw["init_date"])
    frames = []
    for (init_d, g), sub in ec_raw.groupby(["init_date", "group"]):
        sub = sub.sort_values("step")
        if len(sub) < 2:
            continue
        anchor_h = sub["step"].to_numpy(dtype=float)             # 27~51
        target_h = np.arange(28, 52, dtype=float)                # D 01:00 ~ D+1 00:00 (24개)
        u = np.interp(target_h, anchor_h, sub["u10"].to_numpy())
        v = np.interp(target_h, anchor_h, sub["v10"].to_numpy())
        base_utc = init_d + pd.Timedelta(hours=12)
        kst = base_utc + pd.to_timedelta(target_h, unit="h") + pd.Timedelta(hours=9)
        ws = np.hypot(u, v)
        wd = np.arctan2(u, v)
        frames.append(pd.DataFrame({"forecast_kst_dtm": kst, "kpx_group": g,
                                    "ec_ws": ws, "ec_wdsin": np.sin(wd), "ec_wdcos": np.cos(wd)}))
    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates(["forecast_kst_dtm", "kpx_group"], keep="last")
    return out

EC_COLS = ["ec_ws", "ec_wdsin", "ec_wdcos", "ec_dis"]

### 8. no-snap 후처리

In [ ]:
# 7. 후처리 (단순화: 편향보정 + rolling + clip, 스냅 없음)
def postprocess_simple(cf, cap, ws, group, bias_map, clip_lo=0.16, alpha=0.40):
    base = apply_bias_correction_v2(cf * cap, ws, group, bias_map)
    base = apply_rolling_smoothing(pd.Series(base), 3, alpha).to_numpy()
    return np.clip(base, cap * clip_lo, cap)

def score_from_tidy(tidy, pred_col):
    ans, prd = [], []
    for fname, sub in tidy.groupby("fold", sort=False):
        idx = np.sort(sub["forecast_kst_dtm"].unique())
        out = []
        for val in ("target", pred_col):
            m = sub.pivot_table(index="forecast_kst_dtm", columns="group", values=val)
            m.columns = [f"kpx_group_{int(c)}" for c in m.columns]
            out.append(m.reindex(index=idx, columns=TARGET_COLS).fillna(0.0))
        ans.append(out[0]); prd.append(out[1])
    return metric(pd.concat(ans, ignore_index=True), pd.concat(prd, ignore_index=True))

def build_slices(cache):
    return [((fn, int(g)), sub.sort_values("forecast_kst_dtm")) for (fn, g), sub in cache.groupby(["fold", "group"], sort=False)]

### 9. SCADA 가용성 가중치

In [ ]:
# 8. SCADA 가용성
GROUP_TURBINES = {
    1: [("vestas", i) for i in range(1, 7)],
    2: [("vestas", i) for i in range(7, 13)],
    3: [("unison", i) for i in range(1, 6)],
}

def load_scada():
    v = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
    u = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
    return v.merge(u, on="kst_dtm", how="outer").sort_values("kst_dtm").reset_index(drop=True)

def compute_availability(sc):
    sc = sc.copy()
    # FIX: floor -> ceil. 팀원 검증(SCADA합 vs kpx_group_3 라벨 상관): floor=0.949, ceil=0.997.
    # 라벨이 "종료 시각" 기준으로 타임스탬프가 찍혀있다는 뜻 — floor로 묶으면 가용성이
    # 엉뚱한 시간대(한 시간 어긋난 forecast_kst_dtm)에 붙어서 가중치가 새고 있었음.
    sc["hour"] = sc["kst_dtm"].dt.ceil("h")
    rows = []
    for g, turbines in GROUP_TURBINES.items():
        p_cols = [f"{mk}_wtg{i:02d}_power_kw10m" for mk, i in turbines]
        w_cols = [f"{mk}_wtg{i:02d}_ws" for mk, i in turbines]
        grp = sc[["hour"] + p_cols + w_cols]
        med_ws = grp[w_cols].median(axis=1)
        agg = {}
        for (mk, i), pc, wc in zip(turbines, p_cols, w_cols):
            ws_eff = grp[wc].fillna(med_ws)
            windy = ws_eff >= 5.0
            # FIX: 결측(WTG02 2024-02형, 수집 장애 가능성)과 확정 정지(WTG01 2024-12형,
            # 출력 0 실측)를 구분. 기존엔 결측도 확정 정지와 동일하게 취급해 실제로는
            # 살아있었을 수도 있는 구간까지 과도하게 저가중치 처리했음.
            is_missing = grp[pc].isna()
            prod = grp[pc] > 1.0
            dead_strict = (~prod.fillna(False)) & windy & ~is_missing
            missing_windy = is_missing & windy
            slots = grp.groupby("hour")[pc].size()
            agg[f"ds_{mk}{i}"] = dead_strict.groupby(grp["hour"]).sum()
            agg[f"ms_{mk}{i}"] = missing_windy.groupby(grp["hour"]).sum()
            agg[f"n_{mk}{i}"] = slots
        A = pd.DataFrame(agg)
        n_t = len(turbines)
        down_confirmed = sum((A[f"ds_{mk}{i}"] >= np.maximum(A[f"n_{mk}{i}"] - 1, 5)).astype(int) for mk, i in turbines)
        down_missing = sum((A[f"ms_{mk}{i}"] >= np.maximum(A[f"n_{mk}{i}"] - 1, 5)).astype(int) for mk, i in turbines)
        un_s = down_confirmed + 0.5 * down_missing  # 결측은 절반 가중치만 "정지"로 반영 (불확실성 인정)
        rows.append(pd.DataFrame({"forecast_kst_dtm": A.index, "group": g, "avail_strict": 1 - un_s / n_t}))
    return pd.concat(rows, ignore_index=True)

# NOTE: UNISON WTG01/03/04/05가 2023-12부터 출력값이 100kW 단위로 이산화되는 현상을 발견해서
# (팀원 지적 + 검증: 대조군 WTG02는 불변인데 해당 터빈들만 "풍속>=5인데 출력0" 비율이
# 5.8%->14.8%로 급증) 이산화 이후 출력0을 결측과 동일하게 절반 가중치로 낮춰봤으나
# 실측 결과 CV -0.0008 / HOLD 변화없음으로 이득이 없어 되돌림 (2단계: ceil정렬+결측분리까지만 유지).

def avail_factor(av_vals, low=0.05):
    a = np.nan_to_num(av_vals.astype(float), nan=1.0)
    return np.where(a < 0.9, low, np.where(a < 0.999, 0.3, 1.0))

def make_sample_weight_scada(y, capacity, av_series, low_weight=0.05):
    base_w = make_sample_weight(y, capacity, low_weight)
    af = avail_factor(av_series.to_numpy())
    return base_w * af

### 10. 269개 환경 피처 준비

In [ ]:
_, _, _, sample_submission, info = build_datasets()
train_df_tps, test_df_tps, cols_tps, _ = build_v11(info)

spd, spd_cols = build_spd()
train_df_tps = train_df_tps.merge(spd, on="forecast_kst_dtm", how="left")
test_df_tps = test_df_tps.merge(spd, on="forecast_kst_dtm", how="left")

jma_raw = fetch_jma_prevruns(info)
jma = build_jma_frame(jma_raw)
train_df_tps = train_df_tps.merge(jma, on=["forecast_kst_dtm", "kpx_group"], how="left")
test_df_tps = test_df_tps.merge(jma, on=["forecast_kst_dtm", "kpx_group"], how="left")
for frame in [train_df_tps, test_df_tps]:
    frame["jgsm_dis"] = (frame["jgsm_ws"] - frame["ldaps_ws_10m"]).abs()
    frame["jmsm_dis"] = (frame["jmsm_ws"] - frame["ldaps_ws_10m"]).abs()

cols_gspd = cols_tps + JG_COLS + JM_COLS + spd_cols
ec_raw = fetch_ecmwf_prevruns(info)
ec = build_ec_frame(ec_raw)
train_df_tps = train_df_tps.merge(ec, on=["forecast_kst_dtm", "kpx_group"], how="left")
test_df_tps = test_df_tps.merge(ec, on=["forecast_kst_dtm", "kpx_group"], how="left")
for frame in [train_df_tps, test_df_tps]:
    frame["ec_dis"] = (frame["ec_ws"] - frame["ldaps_ws_10m"]).abs()

cols_c0 = cols_gspd + EC_COLS

WEAK_FEATURES = [
    'disagreement_x_season',
    'gfs_wd_cos_100m',
    'gfs_wd_cos_pbl',
    'high_variance_season',
    'jgsm_dis',
    'ldaps_etc_0_VLCDC',
    'ldaps_etc_0_lcc',
    'ldaps_heightAboveGround_10_10u',
    'ldaps_surface_0_SNOM',
    'ldaps_surface_0_lsm',
    'ldaps_surface_0_sp',
    'ldaps_wd_cos_10m',
    'ldaps_ws_10m',
    'phy_gust_over_ws',
    'phy_gust_range',
    'sdG_gust_g5',
    'sdG_t850_g5',
    'sdG_v100_g6',
    'sdG_ws100_g3',
    'sdG_ws10_g6',
    'sdG_ws10_g7',
    'sdG_ws10_g8',
    'sdG_ws10_g9',
    'sdL_blh_g1',
    'sdL_blh_g10',
    'sdL_blh_g11',
    'sdL_blh_g13',
    'sdL_blh_g14',
    'sdL_blh_g15',
    'sdL_blh_g16',
    'sdL_blh_g2',
    'sdL_blh_g3',
    'sdL_blh_g4',
    'sdL_blh_g5',
    'sdL_blh_g7',
    'sdL_blh_g8',
    'sdL_blh_g9',
    'sdL_t2_g15',
    'sdL_t2_g16',
    'sdL_t2_g2',
    'sdL_u10_g10',
    'sdL_u10_g14',
    'sdL_u10_g15',
    'sdL_u10_g16',
    'sdL_u10_g2',
    'sdL_u10_g5',
    'sdL_u10_g8',
    'sdL_v10_g10',
    'sdL_v10_g11',
    'sdL_v10_g12',
    'sdL_v10_g13',
    'sdL_v10_g14',
    'sdL_v10_g16',
    'sdL_v10_g2',
    'sdL_v10_g4',
    'sdL_v10_g7',
    'sdL_v10_g9',
    'sdL_ws10_g1',
    'sdL_ws10_g15',
    'sdL_ws10_g2',
    'sdL_ws10_g5',
    'sdL_ws10_g8',
    'sdL_ws50x_g1',
    'sdL_ws50x_g14',
    'sdL_ws50x_g5',
    'sdL_wsbl_g10',
    'sdL_wsbl_g11',
    'sdL_wsbl_g13',
    'sdL_wsbl_g14',
    'sdL_wsbl_g16',
    'sdL_wsbl_g7',
    'sdL_wsbl_g8',
    'spa_l10_range',
    'spa_l10_std',
    'tmp_g100_d1',
    'tmp_g100_dm1',
    'tmp_g100_dm3',
    'tmp_g100_r3_std',
    'tmp_l10_d1',
    'tmp_l10_d3',
    'tmp_l10_dm1',
    'tmp_l10_r3_std',
    'tmp_l10_r7_std',
]

cols_gspd = [c for c in cols_gspd if c not in WEAK_FEATURES]
cols_c0 = [c for c in cols_c0 if c not in WEAK_FEATURES]



assert len(cols_c0) == 269, f"C0 피처 수가 예상과 다름: {len(cols_c0)}"

sc = load_scada()
av = compute_availability(sc)
av["forecast_kst_dtm"] = pd.to_datetime(av["forecast_kst_dtm"])
availability = av.rename(columns={"group": "kpx_group"})[
    ["forecast_kst_dtm", "kpx_group", "avail_strict"]
]
train_df_tps = train_df_tps.merge(
    availability, on=["forecast_kst_dtm", "kpx_group"], how="left"
)
train_df_tps["avail_strict"] = train_df_tps["avail_strict"].fillna(1.0)
test_df_tps["avail_strict"] = 1.0

assert not train_df_tps.duplicated(["kpx_group", "forecast_kst_dtm"]).any()
assert not test_df_tps.duplicated(["kpx_group", "forecast_kst_dtm"]).any()
print("train:", train_df_tps.shape, "/ test:", test_df_tps.shape)
print("GBM features:", len(cols_c0))

### 11. 다양성 모델·anchor 설정

In [ ]:
if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/BARAM2026")
else:
    DRIVE_ROOT = Path.cwd() / "runs"

ANCHOR_ROOT = DRIVE_ROOT / "hyejin_anen_residual_champion"
ANCHOR_CACHE_DIR = ANCHOR_ROOT / "cache" / "pd16_v1"
ANCHOR_OOF_PATH = ANCHOR_ROOT / "oof_anen_residual.parquet"
ANCHOR_SUBMISSION_PATH = ANCHOR_ROOT / "submission_anen_residual_pd16.csv"

DIVERSITY_VERSION = "deepset_q5_v1"
DIVERSITY_CACHE_DIR = CACHE_DIR / DIVERSITY_VERSION
DIVERSITY_CACHE_DIR.mkdir(parents=True, exist_ok=True)

MEMBER_COLS = [f"mws{k}" for k in range(30)]
CONTEXT_COLS = list(cols_c0)
QUANTILE_LEVELS = [0.10, 0.25, 0.50, 0.75, 0.90]
GROUP_CAPACITY = {1: 21600, 2: 21600, 3: 21000}
GROUP_SCALES = {1: 0.98, 2: 0.98, 3: 0.97}

DIRECT_SEEDS = 3
DIRECT_EPOCHS = 100
DIRECT_PATIENCE = 15
DIRECT_BATCH_SIZE = 1024
ALPHA_GRID = [0.00, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20]
POINT_VARIANTS = ["q50", "q55"]
OPERATION_VARIANTS = ["unscaled", "scaled"]

PASS_MIN_CV_DELTA = 0.0007
PASS_MIN_META_DELTA = 0.0003
PASS_MIN_POSITIVE_FOLDS = 3
PASS_MIN_HOLD_DELTA = -0.0010
PASS_MIN_NMAE_DELTA = -0.0005

assert len(CONTEXT_COLS) == 269
assert len(CONTEXT_COLS) == len(set(CONTEXT_COLS))
assert ANCHOR_OOF_PATH.exists(), (
    "먼저 anen_residual_champion.ipynb의 검증 셀까지 완료하세요: "
    + str(ANCHOR_OOF_PATH)
)

run_config = {
    "experiment": "DEEPSETS_DISTRIBUTION_DIVERSITY",
    "version": DIVERSITY_VERSION,
    "anchor": str(ANCHOR_OOF_PATH),
    "architecture": "permutation-invariant DeepSets plus context encoder",
    "target": "direct capacity factor quantiles",
    "quantiles": QUANTILE_LEVELS,
    "uses_residual_gbm": False,
    "uses_ficr_shift": False,
    "max_partner_weight": max(ALPHA_GRID),
}
(RUN_ROOT / "run_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("anchor:", ANCHOR_ROOT)
print("anchor cache:", ANCHOR_CACHE_DIR)
print("output:", RUN_ROOT)
print("context features:", len(CONTEXT_COLS))

### 12. Anchor OOF와 AnEn 멤버 캐시 정렬

In [ ]:
KEY = ["kpx_group", "forecast_kst_dtm"]


def frame_signature(frame):
    return pd.Timestamp(frame["forecast_kst_dtm"].max()).strftime("%Y%m%d%H%M")


def load_train_members(train_window):
    path = ANCHOR_CACHE_DIR / f"anen_train_{frame_signature(train_window)}.parquet"
    assert path.exists(), f"anchor AnEn train cache 누락: {path}"
    members = pd.read_parquet(path)
    assert members[MEMBER_COLS].notna().all().all()
    return members


def load_query_members(fold_name, query_frame):
    cache_name = "hold_2024full" if fold_name == "fold_2024Q1" else fold_name
    path = ANCHOR_CACHE_DIR / f"anen_query_{cache_name}.parquet"
    assert path.exists(), f"anchor AnEn query cache 누락: {path}"
    members = pd.read_parquet(path)
    keys = query_frame[KEY]
    selected = keys.merge(members, on=KEY, how="left", validate="one_to_one")
    assert len(selected) == len(query_frame)
    assert selected[MEMBER_COLS].notna().all().all()
    return selected


def merge_members(frame, members):
    merged = frame.merge(members[KEY + MEMBER_COLS], on=KEY, how="inner", validate="one_to_one")
    assert len(merged) == len(frame)
    assert merged[MEMBER_COLS].notna().all().all()
    return merged


anchor_oof = pd.read_parquet(ANCHOR_OOF_PATH)
anchor_oof["forecast_kst_dtm"] = pd.to_datetime(anchor_oof["forecast_kst_dtm"])
required_anchor_cols = {
    "fold", "group", "forecast_kst_dtm", "target", "pred_scaled"
}
assert required_anchor_cols.issubset(anchor_oof.columns)
anchor_oof = anchor_oof[[
    "fold", "group", "forecast_kst_dtm", "target", "pred_scaled"
]].rename(columns={"pred_scaled": "anchor_pred"})
assert anchor_oof[["target", "anchor_pred"]].notna().all().all()
print("anchor OOF:", anchor_oof.shape)

### 13. Permutation-invariant DeepSets 확률회귀

In [ ]:
class DeepSetQuantileNet(nn.Module):
    def __init__(self, n_context, hidden=384, p_drop=0.15):
        super().__init__()
        self.member_encoder = nn.Sequential(
            nn.Linear(1, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
        )
        self.member_attention = nn.Linear(64, 1)
        self.context_encoder = nn.Sequential(
            nn.Linear(n_context, 256), nn.BatchNorm1d(256), nn.SiLU(),
            nn.Dropout(p_drop), nn.Linear(256, 128), nn.SiLU(),
        )
        self.group_embedding = nn.Embedding(3, 8)
        self.trunk = nn.Sequential(
            nn.Linear(64 * 4 + 128 + 8, hidden), nn.BatchNorm1d(hidden),
            nn.SiLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden),
            nn.SiLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden),
            nn.SiLU(), nn.Dropout(p_drop),
        )
        self.quantile_head = nn.Linear(hidden, len(QUANTILE_LEVELS))

    def forward(self, members, context, group):
        encoded = self.member_encoder(members.unsqueeze(-1))
        pooled_mean = encoded.mean(dim=1)
        pooled_std = encoded.std(dim=1, unbiased=False)
        pooled_max = encoded.max(dim=1).values
        attention = torch.softmax(self.member_attention(encoded).squeeze(-1), dim=1)
        pooled_attention = (encoded * attention.unsqueeze(-1)).sum(dim=1)
        set_representation = torch.cat(
            [pooled_mean, pooled_std, pooled_max, pooled_attention], dim=1
        )
        context_representation = self.context_encoder(context)
        hidden = self.trunk(torch.cat([
            set_representation, context_representation,
            self.group_embedding(group),
        ], dim=1))
        raw = self.quantile_head(hidden)
        return torch.sort(raw, dim=1).values


QUANTILE_TENSOR = torch.tensor(QUANTILE_LEVELS, dtype=torch.float32, device=DEVICE)


def distribution_loss(prediction, target, weight):
    error = target[:, None] - prediction
    pinball = torch.maximum(
        QUANTILE_TENSOR[None, :] * error,
        (QUANTILE_TENSOR[None, :] - 1.0) * error,
    ).mean(dim=1)
    median_error = prediction[:, 2] - target
    abs_error = torch.abs(median_error)
    huber = torch.where(
        abs_error <= 0.05,
        0.5 * median_error ** 2,
        0.05 * (abs_error - 0.025),
    )
    row_loss = 0.45 * pinball + 0.55 * huber
    return (row_loss * weight).sum() / weight.sum()


def seasonal_es_mask(frame):
    labeled = frame["target"].notna()
    dt = frame["forecast_kst_dtm"]
    last_year = int((dt.loc[labeled] - pd.Timedelta(hours=1)).dt.year.max())
    mask = (
        labeled
        & (dt.dt.year == last_year)
        & dt.dt.month.isin((3, 6, 9, 12))
        & dt.dt.day.isin(range(11, 21))
    )
    if int(mask.sum()) < 240:
        times = np.sort(dt.loc[labeled].unique())
        n_times = min(24 * 40, max(24 * 7, len(times) // 10))
        mask = labeled & (dt >= times[-n_times])
    assert mask.any() and (labeled & ~mask).any()
    return mask


def fit_preprocessors(fit_frame):
    context_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    context_scaler = StandardScaler()
    context_scaler.fit(
        context_imputer.fit_transform(fit_frame[CONTEXT_COLS].to_numpy(dtype=np.float32))
    )
    members = fit_frame[MEMBER_COLS].to_numpy(dtype=np.float32)
    member_mean = float(np.nanmean(members))
    member_std = max(float(np.nanstd(members)), 1e-6)
    return context_imputer, context_scaler, member_mean, member_std


def make_arrays(frame, preprocessors, require_target):
    context_imputer, context_scaler, member_mean, member_std = preprocessors
    members = (
        frame[MEMBER_COLS].to_numpy(dtype=np.float32) - member_mean
    ) / member_std
    context = context_scaler.transform(
        context_imputer.transform(frame[CONTEXT_COLS].to_numpy(dtype=np.float32))
    ).astype(np.float32)
    groups = frame["kpx_group"].map({1: 0, 2: 1, 3: 2}).to_numpy(dtype=np.int64)
    if not require_target:
        return members, context, groups
    capacity = frame["kpx_group"].map(GROUP_CAPACITY).to_numpy(dtype=float)
    target = (frame["target"].to_numpy(dtype=float) / capacity).astype(np.float32)
    low_weight = np.where(target < 0.10, 0.05, 1.0)
    availability_weight = avail_factor(frame["avail_strict"].to_numpy())
    weight = (low_weight * availability_weight).astype(np.float32)
    return members, context, groups, target, weight


def tensor_tuple(arrays):
    members, context, groups, target, weight = arrays
    return (
        torch.tensor(members, device=DEVICE),
        torch.tensor(context, device=DEVICE),
        torch.tensor(groups, device=DEVICE),
        torch.tensor(target, device=DEVICE),
        torch.tensor(weight, device=DEVICE),
    )


def train_direct_model(fit_arrays, es_arrays, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = DeepSetQuantileNet(len(CONTEXT_COLS)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=DIRECT_EPOCHS
    )
    members, context, groups, target, weight = tensor_tuple(fit_arrays)
    es_members, es_context, es_groups, es_target, es_weight = tensor_tuple(es_arrays)
    best_loss, best_epoch, bad_epochs, best_state = float("inf"), 0, 0, None

    for epoch in range(DIRECT_EPOCHS):
        model.train()
        permutation = torch.randperm(len(members), device=DEVICE)
        for start in range(0, len(members), DIRECT_BATCH_SIZE):
            idx = permutation[start:start + DIRECT_BATCH_SIZE]
            optimizer.zero_grad()
            prediction = model(members[idx], context[idx], groups[idx])
            loss = distribution_loss(prediction, target[idx], weight[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            es_prediction = model(es_members, es_context, es_groups)
            validation_loss = distribution_loss(
                es_prediction, es_target, es_weight
            ).item()
        if validation_loss < best_loss - 1e-5:
            best_loss = validation_loss
            best_epoch = epoch + 1
            bad_epochs = 0
            best_state = {
                key: value.detach().clone() for key, value in model.state_dict().items()
            }
        else:
            bad_epochs += 1
            if bad_epochs >= DIRECT_PATIENCE:
                break
    assert best_state is not None
    model.load_state_dict(best_state)
    return model, best_epoch, best_loss


@torch.no_grad()
def predict_direct_model(model, arrays, batch_size=8192):
    members, context, groups = arrays
    model.eval()
    output = []
    for start in range(0, len(members), batch_size):
        stop = start + batch_size
        output.append(model(
            torch.tensor(members[start:stop], device=DEVICE),
            torch.tensor(context[start:stop], device=DEVICE),
            torch.tensor(groups[start:stop], device=DEVICE),
        ).cpu().numpy())
    return np.concatenate(output, axis=0)


DIRECT_EPOCH_ROWS = []


def direct_predict(train_frame, query_frame, tag):
    validation_mask = seasonal_es_mask(train_frame)
    fit_frame = train_frame.loc[~validation_mask].copy()
    es_frame = train_frame.loc[validation_mask].copy()
    preprocessors = fit_preprocessors(fit_frame)
    fit_arrays = make_arrays(fit_frame, preprocessors, require_target=True)
    es_arrays = make_arrays(es_frame, preprocessors, require_target=True)
    query_arrays = make_arrays(query_frame, preprocessors, require_target=False)

    predictions = []
    for seed in range(DIRECT_SEEDS):
        model, best_epoch, best_loss = train_direct_model(
            fit_arrays, es_arrays, seed
        )
        predictions.append(predict_direct_model(model, query_arrays))
        DIRECT_EPOCH_ROWS.append({
            "tag": tag, "seed": seed, "best_epoch": best_epoch,
            "best_loss": best_loss, "n_fit": len(fit_frame),
            "n_es": len(es_frame), "n_query": len(query_frame),
        })
        del model
        torch.cuda.empty_cache()
    prediction = np.mean(predictions, axis=0)
    prediction.sort(axis=1)
    print(tag, "best epochs:", [row["best_epoch"] for row in DIRECT_EPOCH_ROWS[-3:]])
    return prediction

### 14. 4-fold·HOLD 직접 확률예측

In [ ]:
FOLD_MAP = {name: fold for name, fold in FOLD_LIST}


def predict_fold(fold_name):
    path = DIVERSITY_CACHE_DIR / f"direct_{fold_name}.parquet"
    if path.exists():
        cached = pd.read_parquet(path)
        assert cached[["q10", "q25", "q50", "q75", "q90"]].notna().all().all()
        print("direct cache:", path.name, cached.shape)
        return cached

    fold = FOLD_MAP[fold_name]
    train_window = train_df_tps.loc[
        train_df_tps["forecast_kst_dtm"] <= fold["train_end"]
    ].copy()
    valid_window = train_df_tps.loc[
        (train_df_tps["forecast_kst_dtm"] >= fold["valid_start"])
        & (train_df_tps["forecast_kst_dtm"] <= fold["valid_end"])
        & train_df_tps["target"].notna()
    ].copy()
    train_members = load_train_members(train_window)
    valid_members = load_query_members(fold_name, valid_window)
    train_model_frame = merge_members(
        train_window.loc[train_window["target"].notna()].copy(), train_members
    )
    valid_model_frame = merge_members(valid_window, valid_members)
    quantiles = direct_predict(train_model_frame, valid_model_frame, fold_name)
    output = valid_model_frame[KEY + ["target", "ldaps_ws_10m"]].copy()
    output["group"] = output.pop("kpx_group")
    for index, column in enumerate(["q10", "q25", "q50", "q75", "q90"]):
        output[column] = quantiles[:, index]
    output["fold"] = fold_name
    output.to_parquet(path, index=False)
    if DIRECT_EPOCH_ROWS:
        pd.DataFrame(DIRECT_EPOCH_ROWS).drop_duplicates(
            ["tag", "seed"], keep="last"
        ).to_csv(RUN_ROOT / "best_epochs.csv", index=False)
    return output


# Q1과 HOLD는 학습창이 같으므로 HOLD 모델의 2024 전체 예측을 Q1에 재사용합니다.
direct_parts = [
    predict_fold("hold_2024full"),
    predict_fold("fold_2024Q2"),
    predict_fold("fold_2024Q3"),
    predict_fold("fold_2024Q4"),
]
hold_direct = direct_parts[0]
q1_fold = FOLD_MAP["fold_2024Q1"]
q1_direct = hold_direct.loc[
    (hold_direct["forecast_kst_dtm"] >= q1_fold["valid_start"])
    & (hold_direct["forecast_kst_dtm"] <= q1_fold["valid_end"])
].copy()
q1_direct["fold"] = "fold_2024Q1"
direct_parts.append(q1_direct)
direct_oof = pd.concat(direct_parts, ignore_index=True)


def add_partner_predictions(frame):
    parts = []
    for (fold_name, group), group_frame in frame.groupby(["fold", "group"], sort=False):
        group_frame = group_frame.sort_values("forecast_kst_dtm").copy()
        capacity = GROUP_CAPACITY[int(group)]
        group_frame["q55"] = group_frame["q50"] + 0.20 * (
            group_frame["q75"] - group_frame["q50"]
        )
        for point in POINT_VARIANTS:
            raw = group_frame[point].to_numpy(dtype=float) * capacity
            smoothed = apply_rolling_smoothing(
                pd.Series(raw), window=3, alpha=0.40
            ).to_numpy()
            unscaled = np.clip(smoothed, capacity * 0.16, capacity)
            group_frame[f"partner_{point}_unscaled"] = unscaled
            group_frame[f"partner_{point}_scaled"] = np.clip(
                unscaled * GROUP_SCALES[int(group)], 0, capacity
            )
        parts.append(group_frame)
    return pd.concat(parts, ignore_index=True)


direct_oof = add_partner_predictions(direct_oof)
model_frame = anchor_oof.merge(
    direct_oof,
    on=["fold", "group", "forecast_kst_dtm"],
    how="inner", validate="one_to_one", suffixes=("_anchor", "_direct"),
)
assert len(model_frame) == len(anchor_oof) == len(direct_oof)
assert np.allclose(
    model_frame["target_anchor"], model_frame["target_direct"], equal_nan=True
)
model_frame = model_frame.rename(columns={"target_anchor": "target"}).drop(
    columns="target_direct"
)
essential_oof_columns = [
    "target", "anchor_pred", "q10", "q25", "q50", "q75", "q90",
] + [
    f"partner_{point}_{operation}"
    for point in POINT_VARIANTS
    for operation in OPERATION_VARIANTS
]
assert model_frame[essential_oof_columns].notna().all().all()
model_frame.to_parquet(RUN_ROOT / "oof_deepset_diversity.parquet", index=False)
print("diversity OOF:", model_frame.shape)

### 15. Nested OOF 다양성 blend 선택

In [ ]:
CV_FOLDS = [fold["name"] for fold in FOLDS_Q]
HOLD_NAME = "hold_2024full"


def official_score(frame, prediction):
    scored = frame[["fold", "group", "forecast_kst_dtm", "target"]].copy()
    scored["prediction"] = np.asarray(prediction, dtype=float)
    return score_from_tidy(scored, "prediction")


def candidate_prediction(frame, point, operation, alpha):
    partner = frame[f"partner_{point}_{operation}"].to_numpy(dtype=float)
    anchor = frame["anchor_pred"].to_numpy(dtype=float)
    return anchor + alpha * (partner - anchor)


candidate_specs = []
for point in POINT_VARIANTS:
    for operation in OPERATION_VARIANTS:
        for alpha in ALPHA_GRID:
            if alpha == 0.0 and (point != "q50" or operation != "unscaled"):
                continue
            candidate_specs.append((point, operation, alpha))


def candidate_table(frame, folds):
    selected_frame = frame.loc[frame["fold"].isin(folds)]
    anchor_score = official_score(selected_frame, selected_frame["anchor_pred"])
    rows = []
    for point, operation, alpha in candidate_specs:
        prediction = candidate_prediction(selected_frame, point, operation, alpha)
        result = official_score(selected_frame, prediction)
        fold_deltas = []
        for fold_name in folds:
            fold_frame = selected_frame.loc[selected_frame["fold"] == fold_name]
            fold_result = official_score(
                fold_frame,
                candidate_prediction(fold_frame, point, operation, alpha),
            )
            fold_anchor = official_score(fold_frame, fold_frame["anchor_pred"])
            fold_deltas.append(fold_result[0] - fold_anchor[0])
        rows.append({
            "point": point, "operation": operation, "alpha": alpha,
            "Score": result[0], "1-NMAE": result[1], "FiCR": result[2],
            "delta_Score": result[0] - anchor_score[0],
            "delta_1-NMAE": result[1] - anchor_score[1],
            "delta_FiCR": result[2] - anchor_score[2],
            "positive_folds": int(np.sum(np.asarray(fold_deltas) > 0)),
            "min_fold_delta": float(np.min(fold_deltas)),
        })
    return pd.DataFrame(rows).sort_values(
        ["Score", "delta_1-NMAE"], ascending=False
    ).reset_index(drop=True)


candidate_scores = candidate_table(model_frame, CV_FOLDS)
eligible = candidate_scores.loc[
    (candidate_scores["alpha"] > 0)
    & (candidate_scores["positive_folds"] >= PASS_MIN_POSITIVE_FOLDS)
    & (candidate_scores["delta_1-NMAE"] >= PASS_MIN_NMAE_DELTA)
    & (candidate_scores["delta_FiCR"] > 0)
]
selected_row = (
    eligible.iloc[0] if not eligible.empty
    else candidate_scores.loc[candidate_scores["alpha"] == 0].iloc[0]
)
SELECTED_SPEC = {
    "point": selected_row["point"],
    "operation": selected_row["operation"],
    "alpha": float(selected_row["alpha"]),
}

# Nested leave-one-fold-out: 나머지 3개 fold로 사양을 선택해 남은 fold에 적용합니다.
meta_parts, meta_selection_rows = [], []
for heldout_fold in CV_FOLDS:
    train_folds = [fold for fold in CV_FOLDS if fold != heldout_fold]
    train_table = candidate_table(model_frame, train_folds)
    train_eligible = train_table.loc[
        (train_table["alpha"] > 0)
        & (train_table["positive_folds"] >= 2)
        & (train_table["delta_1-NMAE"] >= PASS_MIN_NMAE_DELTA)
        & (train_table["delta_FiCR"] > 0)
    ]
    chosen = (
        train_eligible.iloc[0] if not train_eligible.empty
        else train_table.loc[train_table["alpha"] == 0].iloc[0]
    )
    heldout = model_frame.loc[model_frame["fold"] == heldout_fold].copy()
    heldout["meta_pred"] = candidate_prediction(
        heldout, chosen["point"], chosen["operation"], float(chosen["alpha"])
    )
    meta_parts.append(heldout)
    meta_selection_rows.append({
        "heldout_fold": heldout_fold,
        "point": chosen["point"], "operation": chosen["operation"],
        "alpha": float(chosen["alpha"]),
        "train_delta_score": float(chosen["delta_Score"]),
    })

meta_oof = pd.concat(meta_parts, ignore_index=True)
meta_anchor_score = official_score(meta_oof, meta_oof["anchor_pred"])
meta_candidate_score = official_score(meta_oof, meta_oof["meta_pred"])

cv_frame = model_frame.loc[model_frame["fold"].isin(CV_FOLDS)].copy()
cv_frame["selected_pred"] = candidate_prediction(
    cv_frame, **SELECTED_SPEC
)
hold_frame = model_frame.loc[model_frame["fold"] == HOLD_NAME].copy()
hold_frame["selected_pred"] = candidate_prediction(
    hold_frame, **SELECTED_SPEC
)
anchor_cv = official_score(cv_frame, cv_frame["anchor_pred"])
candidate_cv = official_score(cv_frame, cv_frame["selected_pred"])
anchor_hold = official_score(hold_frame, hold_frame["anchor_pred"])
candidate_hold = official_score(hold_frame, hold_frame["selected_pred"])

fold_rows = []
for fold_name in CV_FOLDS:
    fold_frame = cv_frame.loc[cv_frame["fold"] == fold_name]
    anchor_result = official_score(fold_frame, fold_frame["anchor_pred"])
    candidate_result = official_score(fold_frame, fold_frame["selected_pred"])
    fold_rows.append({
        "fold": fold_name,
        "anchor_score": anchor_result[0],
        "candidate_score": candidate_result[0],
        "delta_score": candidate_result[0] - anchor_result[0],
    })
fold_scores = pd.DataFrame(fold_rows)
positive_folds = int((fold_scores["delta_score"] > 0).sum())

partner_column = f"partner_{SELECTED_SPEC['point']}_{SELECTED_SPEC['operation']}"
partner_correlation = float(
    cv_frame["anchor_pred"].corr(cv_frame[partner_column])
)
cv_delta = float(candidate_cv[0] - anchor_cv[0])
meta_delta = float(meta_candidate_score[0] - meta_anchor_score[0])
hold_delta = float(candidate_hold[0] - anchor_hold[0])

DIVERSITY_GATE_PASS = bool(
    SELECTED_SPEC["alpha"] > 0
    and cv_delta >= PASS_MIN_CV_DELTA
    and meta_delta >= PASS_MIN_META_DELTA
    and positive_folds >= PASS_MIN_POSITIVE_FOLDS
    and hold_delta >= PASS_MIN_HOLD_DELTA
    and candidate_cv[1] - anchor_cv[1] >= PASS_MIN_NMAE_DELTA
    and candidate_cv[2] - anchor_cv[2] > 0
)

summary = pd.DataFrame([
    {"partition": "cv", "model": "anchor", "Score": anchor_cv[0],
     "1-NMAE": anchor_cv[1], "FiCR": anchor_cv[2]},
    {"partition": "cv", "model": "diversity_blend", "Score": candidate_cv[0],
     "1-NMAE": candidate_cv[1], "FiCR": candidate_cv[2]},
    {"partition": "meta_cv", "model": "anchor", "Score": meta_anchor_score[0],
     "1-NMAE": meta_anchor_score[1], "FiCR": meta_anchor_score[2]},
    {"partition": "meta_cv", "model": "diversity_blend", "Score": meta_candidate_score[0],
     "1-NMAE": meta_candidate_score[1], "FiCR": meta_candidate_score[2]},
    {"partition": "hold", "model": "anchor", "Score": anchor_hold[0],
     "1-NMAE": anchor_hold[1], "FiCR": anchor_hold[2]},
    {"partition": "hold", "model": "diversity_blend", "Score": candidate_hold[0],
     "1-NMAE": candidate_hold[1], "FiCR": candidate_hold[2]},
])

gate_decision = {
    "passed": DIVERSITY_GATE_PASS,
    "recommendation": "SUBMIT" if DIVERSITY_GATE_PASS else "DO_NOT_SUBMIT",
    "selected_spec": SELECTED_SPEC,
    "cv_delta_score": cv_delta,
    "meta_delta_score": meta_delta,
    "hold_delta_score": hold_delta,
    "cv_delta_1_nmae": float(candidate_cv[1] - anchor_cv[1]),
    "cv_delta_ficr": float(candidate_cv[2] - anchor_cv[2]),
    "positive_cv_folds": positive_folds,
    "partner_anchor_correlation": partner_correlation,
    "thresholds": {
        "min_cv_delta": PASS_MIN_CV_DELTA,
        "min_meta_delta": PASS_MIN_META_DELTA,
        "min_positive_folds": PASS_MIN_POSITIVE_FOLDS,
        "min_hold_delta": PASS_MIN_HOLD_DELTA,
        "min_nmae_delta": PASS_MIN_NMAE_DELTA,
    },
}

candidate_scores.to_csv(RUN_ROOT / "candidate_scores.csv", index=False)
pd.DataFrame(meta_selection_rows).to_csv(RUN_ROOT / "meta_selection.csv", index=False)
summary.to_csv(RUN_ROOT / "summary.csv", index=False)
fold_scores.to_csv(RUN_ROOT / "fold_scores.csv", index=False)
(RUN_ROOT / "gate_decision.json").write_text(
    json.dumps(gate_decision, ensure_ascii=False, indent=2), encoding="utf-8"
)

display(candidate_scores.head(15).round(6))
display(pd.DataFrame(meta_selection_rows).round(6))
display(summary.round(6))
display(fold_scores.round(6))
print("Gate:", "PASS" if DIVERSITY_GATE_PASS else "REJECT")
print(json.dumps(gate_decision, ensure_ascii=False, indent=2))

### 16. Gate 기반 최종 DeepSets·blend 생성

In [ ]:
FINAL_QUANTILE_PATH = DIVERSITY_CACHE_DIR / "final_test_quantiles.parquet"
PARTNER_SUBMISSION_PATH = RUN_ROOT / "submission_deepset_partner_diagnostic.csv"
BLEND_SUBMISSION_PATH = RUN_ROOT / "submission_deepset_diversity_blend.csv"


def build_final_quantiles():
    if FINAL_QUANTILE_PATH.exists():
        cached = pd.read_parquet(FINAL_QUANTILE_PATH)
        assert len(cached) == len(test_df_tps)
        print("final direct cache:", cached.shape)
        return cached

    train_window = train_df_tps.copy()
    train_members = load_train_members(train_window)
    test_member_path = ANCHOR_CACHE_DIR / "anen_query_final_test.parquet"
    assert test_member_path.exists(), (
        "anchor의 최종 학습 셀을 먼저 완료해야 합니다: " + str(test_member_path)
    )
    test_members = pd.read_parquet(test_member_path)
    train_model_frame = merge_members(
        train_window.loc[train_window["target"].notna()].copy(), train_members
    )
    test_model_frame = merge_members(test_df_tps.copy(), test_members)
    quantiles = direct_predict(train_model_frame, test_model_frame, "final_test")
    output = test_model_frame[[
        "forecast_id", "forecast_kst_dtm", "kpx_group", "ldaps_ws_10m"
    ]].copy()
    for index, column in enumerate(["q10", "q25", "q50", "q75", "q90"]):
        output[column] = quantiles[:, index]
    output.to_parquet(FINAL_QUANTILE_PATH, index=False)
    if DIRECT_EPOCH_ROWS:
        pd.DataFrame(DIRECT_EPOCH_ROWS).drop_duplicates(
            ["tag", "seed"], keep="last"
        ).to_csv(RUN_ROOT / "best_epochs.csv", index=False)
    return output


def make_partner_submission(quantile_frame, point, operation):
    frame = quantile_frame.copy()
    frame["q55"] = frame["q50"] + 0.20 * (frame["q75"] - frame["q50"])
    parts = []
    for group in [1, 2, 3]:
        group_frame = frame.loc[frame["kpx_group"] == group].sort_values(
            "forecast_kst_dtm"
        )
        capacity = GROUP_CAPACITY[group]
        raw = group_frame[point].to_numpy(dtype=float) * capacity
        prediction = apply_rolling_smoothing(
            pd.Series(raw), window=3, alpha=0.40
        ).to_numpy()
        prediction = np.clip(prediction, capacity * 0.16, capacity)
        if operation == "scaled":
            prediction = np.clip(prediction * GROUP_SCALES[group], 0, capacity)
        parts.append(pd.DataFrame({
            "forecast_id": group_frame["forecast_id"].to_numpy(),
            "forecast_kst_dtm": group_frame["forecast_kst_dtm"].to_numpy(),
            "kpx_group": group,
            "prediction": prediction,
        }))
    long = pd.concat(parts, ignore_index=True)
    wide = long.pivot(
        index=["forecast_id", "forecast_kst_dtm"],
        columns="kpx_group", values="prediction",
    )
    wide.columns = [f"kpx_group_{int(group)}" for group in wide.columns]
    wide = wide.reset_index()
    wide["forecast_kst_dtm"] = pd.to_datetime(
        wide["forecast_kst_dtm"]
    ).dt.strftime("%Y-%m-%d %H:%M:%S")
    return sample_submission[["forecast_id"]].merge(
        wide, on="forecast_id", how="left", validate="one_to_one"
    )


if not DIVERSITY_GATE_PASS:
    print("Gate REJECT: 직접 모델과 최종 blend를 학습하지 않습니다.")
else:
    assert ANCHOR_SUBMISSION_PATH.exists(), (
        "anchor 제출 파일이 없습니다. 수정된 anen_residual_champion.ipynb의 "
        "최종 셀까지 실행하세요: " + str(ANCHOR_SUBMISSION_PATH)
    )
    anchor_submission = pd.read_csv(
        ANCHOR_SUBMISSION_PATH, encoding="utf-8-sig"
    )
    assert anchor_submission.shape == sample_submission.shape == (8760, 5)
    assert list(anchor_submission.columns) == list(sample_submission.columns)
    assert anchor_submission["forecast_id"].equals(sample_submission["forecast_id"])

    final_quantiles = build_final_quantiles()
    partner_submission = make_partner_submission(
        final_quantiles, SELECTED_SPEC["point"], SELECTED_SPEC["operation"]
    )
    partner_submission.to_csv(
        PARTNER_SUBMISSION_PATH, index=False, encoding="utf-8-sig"
    )

    blend_submission = anchor_submission.copy()
    alpha = SELECTED_SPEC["alpha"]
    delta_rows = []
    for group in [1, 2, 3]:
        column = f"kpx_group_{group}"
        anchor_values = anchor_submission[column].to_numpy(dtype=float)
        partner_values = partner_submission[column].to_numpy(dtype=float)
        blend_values = anchor_values + alpha * (partner_values - anchor_values)
        blend_submission[column] = np.clip(
            blend_values, 0, GROUP_CAPACITY[group]
        )
        delta_rows.append({
            "group": group, "alpha": alpha,
            "anchor_partner_correlation": float(
                np.corrcoef(anchor_values, partner_values)[0, 1]
            ),
            "mean_delta_kwh": float((blend_values - anchor_values).mean()),
            "mean_abs_delta_kwh": float(np.abs(blend_values - anchor_values).mean()),
            "max_abs_delta_kwh": float(np.abs(blend_values - anchor_values).max()),
        })
    blend_submission.to_csv(
        BLEND_SUBMISSION_PATH, index=False, encoding="utf-8-sig"
    )
    pd.DataFrame(delta_rows).to_csv(
        RUN_ROOT / "submission_delta_summary.csv", index=False
    )
    print("Saved:", BLEND_SUBMISSION_PATH)

### 17. 결과·제출 파일 무결성 검증

In [ ]:
required_paths = [
    RUN_ROOT / "run_config.json",
    RUN_ROOT / "candidate_scores.csv",
    RUN_ROOT / "meta_selection.csv",
    RUN_ROOT / "summary.csv",
    RUN_ROOT / "fold_scores.csv",
    RUN_ROOT / "gate_decision.json",
    RUN_ROOT / "oof_deepset_diversity.parquet",
]
assert all(path.exists() for path in required_paths)

if DIVERSITY_GATE_PASS:
    for path in [PARTNER_SUBMISSION_PATH, BLEND_SUBMISSION_PATH]:
        submission = pd.read_csv(path, encoding="utf-8-sig")
        assert submission.shape == sample_submission.shape == (8760, 5)
        assert list(submission.columns) == list(sample_submission.columns)
        assert submission["forecast_id"].equals(sample_submission["forecast_id"])
        assert submission[TARGET_COLS].notna().all().all()
        for column in TARGET_COLS:
            assert submission[column].between(0, CAPACITY_KWH[column]).all()
        print(path.name, submission.shape, "검증 통과")
    print("제출 권장 파일:", BLEND_SUBMISSION_PATH)
else:
    print("Gate REJECT: 마지막 제출권을 사용하지 않습니다.")

print("결과 저장 경로:", RUN_ROOT)
print("최종 판정:", gate_decision["recommendation"])

### 생성 파일

`/content/drive/MyDrive/BARAM2026/hyejin_deepset_distribution_diversity`에 저장됩니다.

- `candidate_scores.csv`: anchor 고정 상태의 point·scale·alpha 후보 비교
- `meta_selection.csv`: leave-one-fold-out에서 실제 선택된 사양
- `summary.csv`: anchor와 diversity blend의 CV·meta CV·HOLD 비교
- `fold_scores.csv`: 선택 후보의 fold별 개선량
- `gate_decision.json`: 마지막 제출 여부
- `best_epochs.csv`: DeepSets seed별 best epoch
- `oof_deepset_diversity.parquet`: anchor·직접 분위수·blend OOF
- `submission_deepset_partner_diagnostic.csv`: 직접 모델 진단용이며 단독 제출 금지
- `submission_deepset_diversity_blend.csv`: gate 통과 시 최종 제출 후보

최종 셀에서 `SUBMIT`일 때만 `submission_deepset_diversity_blend.csv`를 제출합니다.